# Closed-World Re-ID Evaluation (Video-to-Video & Image-to-Video)

This notebook evaluates distance matrices generated by models in the **Closed-World** setting.
It computes and prints:
- **mAP** (mean Average Precision) with 95% Bootstrap Confidence Intervals
- **Rank-1** Cumulative Matching Characteristic with 95% Bootstrap Confidence Intervals
- **Rank-5** Cumulative Matching Characteristic with 95% Bootstrap Confidence Intervals
- **Rank-10** Cumulative Matching Characteristic


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")


Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


In [2]:
# =====================================================================
# Configuration: Models and Candidate CSV Paths
# =====================================================================
MODELS = ["bot", "agw", "arbase", "sbs", "mgn", "tfclip", "dinov2"]
MODALITIES = ["vid2vid", "img2vid"]

SEARCH_DIRS = [
    ROOT_DIR / "evaluation" / "csvs_closed_results"
]

def find_closed_csv(model_name: str, modality: str):
    """Find closed-world distance matrix CSV for a given model and modality."""
    patterns = [
        f"{model_name}_{modality}/closed_dist_matrix.csv",
        f"{model_name}_{modality}/vid2vid_closed_dist_matrix.csv",
        f"{model_name}_{modality}/img2vid_closed_dist_matrix.csv",
        f"{model_name}_closed/closed_dist_matrix.csv",
        f"{model_name}_closed_{modality}/closed_dist_matrix.csv",
    ]
    for search_dir in SEARCH_DIRS:
        if not search_dir.exists():
            continue
        for pat in patterns:
            candidate = search_dir / pat
            if candidate.exists() and candidate.stat().st_size > 1000:
                return candidate
        # Direct folder check
        folder = search_dir / f"{model_name}_{modality}"
        if folder.exists():
            for f in folder.glob("*closed*.csv"):
                if f.stat().st_size > 1000:
                    return f
    return None

models_to_test = {}
for m in MODELS:
    for mod in MODALITIES:
        key = f"{m.upper()} ({mod})"
        csv_path = find_closed_csv(m, mod)
        if csv_path:
            models_to_test[key] = str(csv_path)
            print(f"[FOUND] {key:<20} -> {csv_path.relative_to(ROOT_DIR)}")
        else:
            print(f"[PENDING] {key:<18} -> Not yet found in search directories")


[FOUND] BOT (vid2vid)        -> evaluation/csvs_closed_results/bot_vid2vid/closed_dist_matrix.csv
[FOUND] BOT (img2vid)        -> evaluation/csvs_closed_results/bot_img2vid/closed_dist_matrix.csv
[FOUND] AGW (vid2vid)        -> evaluation/csvs_closed_results/agw_vid2vid/closed_dist_matrix.csv
[PENDING] AGW (img2vid)      -> Not yet found in search directories
[PENDING] ARBASE (vid2vid)   -> Not yet found in search directories
[PENDING] ARBASE (img2vid)   -> Not yet found in search directories
[PENDING] SBS (vid2vid)      -> Not yet found in search directories
[PENDING] SBS (img2vid)      -> Not yet found in search directories
[PENDING] MGN (vid2vid)      -> Not yet found in search directories
[PENDING] MGN (img2vid)      -> Not yet found in search directories
[PENDING] TFCLIP (vid2vid)   -> Not yet found in search directories
[PENDING] TFCLIP (img2vid)   -> Not yet found in search directories
[PENDING] DINOV2 (vid2vid)   -> Not yet found in search directories
[PENDING] DINOV2 (img2vid)

In [3]:
# =====================================================================
# Run Bootstrap Evaluation
# =====================================================================
M_ITERATIONS = 100
all_closed_results = {}

for name, path in models_to_test.items():
    print("=" * 60)
    print(f"Evaluating {name} on {Path(path).name} ...")
    print("=" * 60)
    try:
        res = bootstrap_from_csv(path, m=M_ITERATIONS, mode="closed")
        all_closed_results[name] = res
    except Exception as e:
        print(f"[ERROR] Failed to evaluate {name}: {e}")


Evaluating BOT (vid2vid) on closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:01<00:00, 55.33it/s]


Evaluating BOT (img2vid) on closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 55.96it/s]


Evaluating AGW (vid2vid) on closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 55.62it/s]


In [4]:
# =====================================================================
# Display Closed-World Benchmark Table
# =====================================================================
rows = []
for name, res in all_closed_results.items():
    full = res.get("full_set", {})
    
    # mAP
    m_mean = res.get("mAP_boot_mean", full.get("mAP", 0.0)) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    m_low, m_high = m_ci[0] * 100, m_ci[1] * 100
    
    # CMC Rank-1, Rank-5, Rank-10
    cmc_boot = res.get("cmc_boot_mean", full.get("cmc", np.zeros(10)))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    
    r1 = cmc_boot[0] * 100
    r1_l, r1_u = cmc_l[0] * 100, cmc_u[0] * 100
    
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_l = cmc_l[4] * 100 if len(cmc_l) >= 5 else 0.0
    r5_u = cmc_u[4] * 100 if len(cmc_u) >= 5 else 0.0
    
    r10 = cmc_boot[9] * 100 if len(cmc_boot) >= 10 else 0.0
    
    rows.append({
        "Model (Modality)": name,
        "mAP (%)": f"{m_mean:.2f} [{m_low:.2f}-{m_high:.2f}]",
        "Rank-1 (%)": f"{r1:.2f} [{r1_l:.2f}-{r1_u:.2f}]",
        "Rank-5 (%)": f"{r5:.2f} [{r5_l:.2f}-{r5_u:.2f}]"
    })

df_closed = pd.DataFrame(rows)

print("=" * 110)
print(f"{"Model (Modality)":<22} | {"mAP [95% CI]":<26} | {"Rank-1 [95% CI]":<26} | {"Rank-5 [95% CI]":<26}")
print("=" * 110)
for r in rows:
    print(f"{r["Model (Modality)"]:<22} | {r["mAP (%)"]:<26} | {r["Rank-1 (%)"]:<26} | {r["Rank-5 (%)"]:<26} ")
print("-" * 110)

display_cols = ["Model (Modality)", "mAP (%)", "Rank-1 (%)", "Rank-5 (%)"]
df_closed[display_cols]


Model (Modality)       | mAP [95% CI]               | Rank-1 [95% CI]            | Rank-5 [95% CI]           
BOT (vid2vid)          | 24.63 [22.59-26.80]        | 24.29 [21.43-26.74]        | 45.57 [42.49-48.84]        
BOT (img2vid)          | 18.67 [17.08-20.28]        | 17.48 [15.38-19.56]        | 35.74 [32.82-38.24]        
AGW (vid2vid)          | 26.98 [24.95-29.50]        | 26.56 [24.51-29.65]        | 49.12 [46.24-52.73]        
--------------------------------------------------------------------------------------------------------------


,Model (Modality),mAP (%),Rank-1 (%),Rank-5 (%)
0,BOT (vid2vid),24.63 [22.59-26.80],24.29 [21.43-26.74],45.57 [42.49-48.84]
1,BOT (img2vid),18.67 [17.08-20.28],17.48 [15.38-19.56],35.74 [32.82-38.24]
2,AGW (vid2vid),26.98 [24.95-29.50],26.56 [24.51-29.65],49.12 [46.24-52.73]
